In [1]:
#!pip install --upgrade transformers

In [21]:
from transformers import pipeline
import json
from pathlib import Path
import re

import pandas as pd

In [3]:
#import transformers
#print(transformers.__version__)

5.4.0


In [4]:
# ── 1. Load model ─────────────────────────────────────────────────────────────
# Good small options to try:
# - "mistralai/Mistral-7B-Instruct-v0.3"
# - "Qwen/Qwen2.5-7B-Instruct"
# - "BioMistral/BioMistral-7B" (medical-focused)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device_map="auto",      # automatically uses GPU if available, else CPU
    max_new_tokens=1024,
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [78]:

# ── 2. Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a medical information extraction assistant.
Given a medical text, extract structured information and return it as JSON only,
with no preamble or explanation.
"""

def build_prompt(text: str) -> str:
    return f"""Extract the following fields from the medical text below.
If a field is not mentioned, use null.

Fields to extract:
- circumstances: the cause of the incident which lead to the ER visit
- symptoms: list of symptoms reported by the patient
- diagnosis: the diagnosis if mentioned
- medications: list of medications mentioned
- patient_age: age of the patient if mentioned
- patient_gender: gender of the patient if mentioned

Return only a JSON object with these fields.

Medical text:
{text}
"""

def build_prompt_normalized(text: str) -> str:
    return f"""Extract clinical information and normalize it using standard medical terminology.

- Map symptoms and diagnoses to standardized terms (e.g., SNOMED-like labels if possible).
- Normalize medications to their generic names.
- Normalize age to integer if possible.

Return JSON with:
- symptoms_normalized
- diagnosis_normalized
- medications_normalized

Medical text:
{text}
"""


def build_prompt_timeline(text: str) -> str:
    return f"""Extract a timeline of events from the medical text.

Return a JSON list of events in chronological order:
- event
- time_expression (exact or relative, e.g., "2 days ago")

Medical text:
{text}
"""

def build_prompt_causality(text: str) -> str:
    return f"""Identify causal relationships in the medical text.

Return JSON:
- cause: what triggered the condition or ER visit
- effect: resulting symptoms or diagnosis
- confidence: high / medium / low

Medical text:
{text}
"""

def build_prompt_negation(text: str) -> str:
    return f"""Extract symptoms and indicate whether they are:
- present
- absent (negated)
- uncertain

Return JSON list:
- symptom
- status (present / absent / uncertain)

Medical text:
{text}
"""

def build_prompt_clinical_summary(text: str) -> str:
    return f"""Summarize the medical text for a physician.

- Use precise clinical language
- Be concise
- Include key findings only

Medical text:
{text}
"""


def build_prompt_patient_summary(text: str) -> str:
    return f"""Explain the medical text to a patient with no medical background.

- Use simple language
- Avoid jargon
- Be reassuring but accurate

Medical text:
{text}
"""

def build_prompt_triage(text: str) -> str:
    return f"""Classify the urgency level of this case.

Return JSON:
- triage_level: (low / moderate / high / emergency)
- justification: short explanation

Medical text:
{text}
"""


def build_prompt_inconsistency(text: str) -> str:
    return f"""Detect inconsistencies or contradictions in the medical text.

Return JSON list:
- statement_1
- statement_2
- explanation

Medical text:
{text}
"""


def build_prompt_missing(text: str) -> str:
    return f"""Identify missing but clinically relevant information.

Return JSON list of missing fields (e.g., allergies, medications, history).

Medical text:
{text}
"""


def build_prompt_relations(text: str) -> str:
    return f"""Extract relationships between entities.

Return JSON list:
- subject
- relation (e.g., "has_symptom", "treated_with")
- object

Medical text:
{text}
"""



def build_prompt_noisy(text: str) -> str:
    return f"""Extract key medical information despite possible noise, typos, or incomplete sentences.

Return JSON:
- symptoms
- diagnosis
- medications

Medical text:
{text}
"""


def build_prompt_counterfactual(text: str) -> str:
    return f"""Based on the medical text, answer what would likely change 
if the patient had NOT received the mentioned treatment.

Return ONLY a JSON object with exactly these fields:
- treatment_received: the treatment mentioned in the text
- likely_outcome_without_treatment: short clinical explanation
- severity_increase: yes / no / uncertain

Medical text:
{text}
"""


def build_prompt_sensitivity(text: str) -> str:
    return f"""Extract diagnosis and symptoms.

Be precise: small wording differences matter.

Return JSON:
- symptoms
- diagnosis

Medical text:
{text}
"""


def build_prompt_cause(text: str) -> str:
    return f"""Identify the precipitating event that led to the medical emergency described in the text.

A precipitating event is a concrete external incident such as: a fall, a car accident, 
domestic violence, a suicide attempt, a sports injury, an overdose, a workplace accident, etc.

Do NOT return a diagnosis or symptom as the event (e.g. "myocardial infarction", "chest pain" are NOT events).
If no such external incident is mentioned, return null.

Example of valid output when event is known:
{{"event": "fall from ladder", "confidence": "high", "explanation": "The patient fell from a ladder."}}

Example of valid output when event is unknown:
{{"event": null, "confidence": "unknown", "explanation": "No precipitating event is clearly stated in the text."}}

Return ONLY a valid JSON object with exactly these fields:
- "event": the precipitating external incident in a few words, or null if not mentioned
- "confidence": "high" / "medium" / "low" / "unknown"
- "explanation": one sentence justifying your answer

Medical text:
{text}
"""


In [98]:


# ── 3. Extraction function ────────────────────────────────────────────────────
def extract_clinical_text(raw_text: str) -> str:

    CLINICAL_START_MARKERS = [
    "CHIEF COMPLAINT:",
    "HISTORY OF PRESENT ILLNESS:",
    "REASON FOR VISIT:",
    "PRESENTING COMPLAINT:",
    "CC:",
        ]
    lines = raw_text.splitlines()
    
    start_idx = None
    for i, line in enumerate(lines):
        if any(line.strip().startswith(marker) for marker in CLINICAL_START_MARKERS):
            start_idx = i
            break
    
    end_idx = None
    for i, line in enumerate(lines):
        if "About This Sample:" in line or "Go Back to" in line:
            end_idx = i
            break

    print(f"  start_idx={start_idx}, end_idx={end_idx}, total_lines={len(lines)}")  # ADD
    
    if start_idx is None:
        return ""
    
    clinical_lines = lines[start_idx:end_idx]
    return "\n".join(l for l in clinical_lines if l.strip())

In [99]:
# ── 4. Process a folder of .txt files ────────────────────────────────────────
# Test on just the first 5 files
def process_er_folder(folder_path: str, limit: int = None) -> pd.DataFrame:
    records = []
    txt_files = sorted(Path(folder_path).glob("*.txt"))
    if limit:
        txt_files = txt_files[:limit]

    print("Files selected:")
    for f in txt_files:
        print(f"  {f.name}")
    print(f"Processing {len(txt_files)} files...")

    for i, txt_file in enumerate(txt_files):
        raw_text = txt_file.read_text(encoding="utf-8").strip()
        text = extract_clinical_text(raw_text)  # call ONCE
        
        print(f"Raw text preview: {repr(raw_text[:100])}")   # preview raw
        print(f"Cleaned text preview: {repr(text[:100])}")   # preview cleaned
    
        if not text:
            print(f"  Skipped: no clinical text found in {txt_file.name}")
            continue
    
        extracted = extract_medical_info(text)  # pass cleaned text
        extracted["source_file"] = txt_file.name
        records.append(extracted)

    print(f"Total records collected: {len(records)}")  # ADD
    return pd.DataFrame(records)


In [100]:
df_test = process_er_folder("mtsamples_er/texts", limit=5)
#print(df_test)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Files selected:
  Abdominal_Pain_-_Consult.txt
  Abdominal_Pain_2D_Consult.txt
  Abrasions_26_Lacerations_2D_ER_Visit.txt
  Abrasions__Lacerations_-_ER_Visit.txt
  Accidental_Celesta_Ingestion_-_ER_Visit.txt
Processing 5 files...
  start_idx=0, end_idx=44, total_lines=206
Raw text preview: 'CHIEF COMPLAINT:\n  Abdominal pain.\nHISTORY OF PRESENT ILLNESS:\n  The patient is a 71-year-old female'
Cleaned text preview: 'CHIEF COMPLAINT:\n  Abdominal pain.\nHISTORY OF PRESENT ILLNESS:\n  The patient is a 71-year-old female'


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  start_idx=0, end_idx=44, total_lines=206
Raw text preview: 'CHIEF COMPLAINT:\n  Abdominal pain.\nHISTORY OF PRESENT ILLNESS:\n  The patient is a 71-year-old female'
Cleaned text preview: 'CHIEF COMPLAINT:\n  Abdominal pain.\nHISTORY OF PRESENT ILLNESS:\n  The patient is a 71-year-old female'


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  start_idx=0, end_idx=66, total_lines=228
Raw text preview: 'HISTORY OF PRESENT ILLNESS: \n This is a 12-year-old male, who was admitted to the Emergency Departme'
Cleaned text preview: 'HISTORY OF PRESENT ILLNESS: \n This is a 12-year-old male, who was admitted to the Emergency Departme'


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  start_idx=0, end_idx=66, total_lines=228
Raw text preview: 'HISTORY OF PRESENT ILLNESS: \n This is a 12-year-old male, who was admitted to the Emergency Departme'
Cleaned text preview: 'HISTORY OF PRESENT ILLNESS: \n This is a 12-year-old male, who was admitted to the Emergency Departme'


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  start_idx=0, end_idx=37, total_lines=189
Raw text preview: 'HISTORY OF PRESENT ILLNESS:\n  Patient is a three years old male who about 45 minutes prior admission'
Cleaned text preview: 'HISTORY OF PRESENT ILLNESS:\n  Patient is a three years old male who about 45 minutes prior admission'
Total records collected: 5


In [90]:
for i, row in df_test.iterrows():
    # Guard against missing source_file column
    source = row.get('source_file', 'unknown')
    print(f"── Record {i+1}: {source} ──")
    print(json.dumps({k: v for k, v in row.items()}, indent=2))
    print()

── Record 1: Abdominal_Pain_-_Consult.txt ──
{
  "event": "Recurrence of Sigmoid Diverticulitis",
  "confidence": "high",
  "explanation": "The patient has a history of sigmoid diverticulitis and the current symptoms (abdominal pain localized to the left lower quadrant) and diagnostic studies (CT scan and consultation with a colorectal surgeon) suggest a recurrence of this condition.",
  "source_file": "Abdominal_Pain_-_Consult.txt"
}

── Record 2: Abdominal_Pain_2D_Consult.txt ──
{
  "event": "Recurrence of sigmoid diverticulitis",
  "confidence": "high",
  "explanation": "The patient has a history of sigmoid diverticulitis and is currently experiencing abdominal pain localized to the left lower quadrant, with a fullness or mass in the same area. Additionally, a repeat stat CT scan of the abdomen and pelvis is recommended as a diagnostic study, and the working diagnosis is sigmoid diverticulitis.",
  "source_file": "Abdominal_Pain_2D_Consult.txt"
}



In [10]:
#!du -sh ~/.cache/huggingface/hub/

In [93]:
raw_text = Path("mtsamples_er/texts/Abrasions_26_Lacerations_2D_ER_Visit.txt").read_text(encoding="utf-8")
first_line = raw_text.splitlines()[0]
print(repr(first_line))

'HISTORY OF PRESENT ILLNESS: '
